# 04_業務_訂單金額

新增計算結果與排序

任務：逐列計算訂單金額＝單價 × 數量。；依訂單金額由大到小排序，列出前 3 筆。；計算全部訂單金額，輸出含計算結果的 CSV。

規則：一列一張單品訂單；單價為本例約定成交價，不另加稅或運費。

先人工核對再執行。資料皆為教學模擬值。

In [ ]:
%pip -q install openpyxl numpy matplotlib

In [ ]:
from pathlib import Path
import csv
from openpyxl import load_workbook

# 本機：把 Notebook 與 Excel 放在同一資料夾。Colab：先上傳本例 Excel。
SOURCE = Path('04_業務_訂單金額.xlsx')
if not SOURCE.exists():
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(f'請將 {SOURCE.name} 放到目前工作目錄：{Path.cwd()}')
    files.upload()
if not SOURCE.exists():
    raise FileNotFoundError(f'尚未找到 {SOURCE.name}，請確認上傳檔名')
OUT = Path('output') / '04_業務_訂單金額'
OUT.mkdir(parents=True, exist_ok=True)

def read_rows(sheet):
    data = sheet.iter_rows(values_only=True)
    headers = next(data)
    return [dict(zip(headers, row)) for row in data if any(v is not None for v in row)]

def save_csv(name, records):
    if not records:
        print('沒有符合條件的資料，未輸出 CSV')
        return
    with (OUT / name).open('w', newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(f, fieldnames=list(records[0]))
        writer.writeheader()
        writer.writerows(records)

# 來源檔只有原始值，沒有需要重算的 Excel 公式。
wb = load_workbook(SOURCE, read_only=True, data_only=True)
print('工作表：', wb.sheetnames)
rows = read_rows(wb['資料'])
print('第一筆：', rows[0])

In [ ]:
result = [dict(r, 訂單金額_元=r['單價_元'] * r['數量']) for r in rows]
result.sort(key=lambda r: (-r['訂單金額_元'], r['訂單編號']))
print('前 3 筆：', result[:3])
print('總額：', sum(r['訂單金額_元'] for r in result))
save_csv('訂單金額.csv', result)
assert sum(r['訂單金額_元'] for r in result) == 18400
assert [r['訂單編號'] for r in result[:3]] == ['S003', 'S004', 'S001']
wb.close()

## 驗收答案

總額 18,400 元；前三名 S003（10,000）、S004（3,200）、S001（2,400）。